In [143]:
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

In [212]:
class Multi_Contact_Solving:
  def __init__(self, data) :
    df1 = pd.read_json(data)
    self.df = df1.replace('', np.NaN)
    self.d = {i: set() for i in self.df.Id}
    self.email_group = None
    self.phone_group = None
    self.order_group = None
    self.id_to_contact = self.df.set_index('Id').Contacts.to_dict()
    self.out_model = None

  def groupby_features(self) :
    self.email_group = self.df.groupby('Email').Id.agg(lambda x: set(x))
    self.phone_group = self.df.groupby('Phone').Id.agg(lambda x: set(x))
    self.order_group = self.df.groupby('OrderId').Id.agg(lambda x: set(x))

  def filter_groupby_values(self):
    for ids in self.email_group:
        for id in ids:
          self.d[id] |= set(ids)

    for ids in self.phone_group:
        for id in ids:
          self.d[id] |= set(ids)

    for ids in self.order_group:
        for id in ids:
          self.d[id] |= set(ids)

  def merge_across_groups_again(self) :
    for i in tqdm(range(6)):
        for id, ids in self.d.items():
            for id_ in list(ids):
                self.d[id] |= self.d[id_]
  
  def group_all_format_submit(self):
    self.df['set'] = self.df.Id.apply(lambda x: self.d[x])
    self.df['trace'] = self.df.set.apply(lambda x: '-'.join(map(str, sorted(list(x)))))
    self.df['n_con'] = self.df.set.apply(lambda x: str([self.id_to_contact[id] for id in x]))
    self.df['out'] = self.df.trace + ', ' + self.df.n_con
    self.out_model = self.df[['Id', 'out']]
    self.out_model.columns = ['ticket_id', 'ticket_trace/contact']
    return self.out_model
    
  def submit_model(self,name_file):
    return self.out_model.to_csv(name_file, index=False)
    
  

In [213]:
tes = Multi_Contact_Solving('contacts.json')

In [214]:
tes.groupby_features()

In [215]:
tes.filter_groupby_values()

In [216]:
tes.merge_across_groups_again()

In [217]:
tes.group_all_format_submit()

,ticket_id,ticket_trace/contact
0,0,"0, [1]"
1,1,"1-2458-98519-115061-140081-165605-476346, [4, ..."
2,2,"2-159312-322639-348955, [1, 0, 0, 3]"
3,3,"3, [0]"
4,4,"4, [2]"
...,...,...
499995,499995,"499995, [2]"
499996,499996,"499996, [4]"
499997,499997,"499997, [2]"
499998,499998,"121111-499998, [1, 4]"


In [218]:
tes.submit_model('submit_model.csv')

In [219]:
df = pd.read_csv('submit_model.csv')
df.head()

,ticket_id,ticket_trace/contact
0,0,"0, [1]"
1,1,"1-2458-98519-115061-140081-165605-476346, [4, ..."
2,2,"2-159312-322639-348955, [1, 0, 0, 3]"
3,3,"3, [0]"
4,4,"4, [2]"
